[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SysBioChalmers/MESBcourse/blob/main/exercises/GEM2_gecko/gem2.ipynb)

# GEM2: enzyme-constrained models in geckopy

A Python / **geckopy** port of the GECKO 3 MATLAB exercise (`gem2_stage*.mlx`).

We **build** an enzyme-constrained model (**ecModel**) of the yeast
*Rhodotorula toruloides* from a conventional GEM and then tune, constrain and
analyse it — the same workflow as the MATLAB exercise, all in Python:

- **Stage 1** — expand the conventional GEM into an ecModel structure.
- **Stage 2** — populate kcat values (BRENDA + DLKcat) and apply the enzyme
  and protein-pool constraints.
- **Stage 3** — tune kcat values so the model reaches the experimental growth.
- **Stage 4** — integrate proteomics data.
- **Stage 5** — simulate: minimal protein usage, ecModel vs conventional GEM.

The reconstruction normally queries online databases (UniProt, BRENDA,
PubChem) and runs DLKcat in Docker; here we use the **cached data files**
shipped with the exercise (`uniprot.tsv`, `DLKcat.tsv`, `smilesDB.tsv`), exactly
as the MATLAB version provides pre-generated files.

> ### ⚠️ Key difference: direction of the protein reactions
> geckopy and the GECKO MATLAB toolbox represent the protein machinery with
> **opposite reaction directions**:
> | | GECKO MATLAB | geckopy |
> |---|---|---|
> | `usage_prot_<id>` | `prot_<id> → prot_pool`, bounds `(-1000, 0)` | `prot_pool → prot_<id>`, bounds `(0, 1000)` |
> | `prot_pool_exchange` | `prot_pool → ∅`, bounds `(-1000, 0)` | `∅ → prot_pool`, bounds `(0, 1000)` |
> | flux sign | **negative** | **positive** |
> | proteomics constraint | set `lb = -conc` | set `ub = conc` |
>
> So in geckopy you **minimize the (positive) `prot_pool_exchange`** for minimal
> protein usage, and protein/usage fluxes are positive.

## Setup

geckopy is not on PyPI yet, so it is installed from GitHub. It depends on
`raven-toolbox`; geckopy's `main` currently needs `raven-toolbox` from its `main`
branch (the pinned `develop` lags behind), so we install that explicitly.

In [ ]:
import sys
!{sys.executable} -m pip install -q "git+https://github.com/SysBioChalmers/geckopy.git@main"
!{sys.executable} -m pip install -q --force-reinstall --no-deps "git+https://github.com/SysBioChalmers/raven-toolbox.git@main"
!{sys.executable} -m pip install -q matplotlib pandas numpy

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geckopy
from geckopy import (
    ModelParameters, ModelAdapter,
    load_conventional_gem, make_ec_model, load_uniprot_tsv,
    fill_eccodes_from_database, fill_eccodes_from_gem,
    fuzzy_kcat_matching, load_brenda_data, find_met_smiles, read_dlkcat_output,
    merge_dlkcat_and_fuzzy_kcats, apply_kcat_list, fill_kcats_from_isozymes,
    assign_standard_kcat, apply_kcat_constraints, set_prot_pool_size,
    sensitivity_tuning, enzyme_usage, report_enzyme_usage, map_rxns_to_conv,
    load_prot_data, load_flux_data, fill_enz_concs, constrain_enz_concs,
    calculate_f_factor, apply_flux_data_constraints, flexibilize_enz_concs,
)
from geckopy.databases.phyl_dist import PhylDist
print('geckopy ready; cobra', __import__('cobra').__version__)

Download the conventional GEM and the cached reconstruction/analysis data.

In [ ]:
os.makedirs('ecRhto/models', exist_ok=True)
os.makedirs('ecRhto/data', exist_ok=True)
base = "https://raw.githubusercontent.com/SysBioChalmers/MESBcourse/main/exercises/GEM2_gecko"
for sub, fname in [('models', 'rhto.xml'),
                   ('data', 'uniprot.tsv'), ('data', 'DLKcat.tsv'), ('data', 'smilesDB.tsv'),
                   ('data', 'abs_proteomics.txt'), ('data', 'fluxData.tsv')]:
    dst = f'ecRhto/{sub}/{fname}'
    if not os.path.exists(dst):
        !wget -q {base}/ecRhto/{sub}/{fname} -O {dst}
print('downloaded:', os.listdir('ecRhto/models'), os.listdir('ecRhto/data'))

## Model adapter

GECKO uses a *model adapter* to hold organism-specific parameters. In MATLAB
this is the `ecRhtoAdapter.m` class; in geckopy it is a `ModelAdapter` built
from `ModelParameters`, with the same values.

In [ ]:
params = ModelParameters(
    path=os.path.abspath('ecRhto'),
    conv_gem=os.path.abspath('ecRhto/models/rhto.xml'),
    org_name='Rhodotorula toruloides',
    sigma=0.5, p_tot=0.4385, f=0.5, gr_exp=0.18,
    c_source='r_1714',   # glucose exchange
    bio_rxn='r_4041',    # biomass pseudoreaction
    enzyme_comp='cytoplasm',
)
adapter = ModelAdapter(params)
gem = load_conventional_gem(adapter)
print(f'Conventional GEM: {len(gem.reactions)} reactions, {len(gem.genes)} genes')

### Question 1 (cf. MATLAB)

What would you do if your organism is not on UniProt at all, or only a few of
its proteins are? (Hint: GECKO needs, per gene, a sequence and a molecular
weight — where else could those come from?)

## Stage 1 — expand the GEM into an ecModel structure

`make_ec_model` splits reactions per isoenzyme, adds a `prot_<id>`
pseudometabolite and a `usage_prot_<id>` reaction for every enzyme, and a
shared `prot_pool`. Enzyme data (sequence, MW) comes from the cached UniProt
table.

In [ ]:
uniprot_db = load_uniprot_tsv('ecRhto/data/uniprot.tsv')
ec = make_ec_model(gem, adapter, uniprot_db=uniprot_db)
print(f'ecModel: {len(ec.reactions)} reactions, {len(ec.ec.enzymes)} enzymes, '
      f'{len(ec.ec.rxns)} enzyme-constrained reactions')

The log lists a few genes not found in UniProt (e.g. the mitochondrially
encoded `COX1/2/3`). Their reactions are left enzyme-unconstrained.

### Question 2 (cf. MATLAB)

Inspect the genes that were not found in UniProt. What does this mean for the
model, and is the impact large or minor? (How many genes, and how would you
resolve it?)

### Question 3 (cf. MATLAB)

`ec.ec.kcat` has the same length as `ec.ec.rxns`, not `ec.ec.enzymes` — kcats
are stored **per reaction**, not per enzyme. Why? And which field still
connects enzymes to reactions? (Look at `ec.ec.rxn_enz_mat`.)

## Stage 2 — populate kcat values and apply constraints

### EC numbers

EC numbers are gathered from UniProt and from the GEM's own annotations.

In [ ]:
fill_eccodes_from_database(ec, uniprot_db)   # from UniProt
fill_eccodes_from_gem(ec)                       # fill the rest from the GEM
no_ec = sum(1 for e in ec.ec.eccodes if not e)
print(f'{no_ec} of {len(ec.ec.rxns)} enzyme-constrained reactions still have no EC number')

### kcat from BRENDA (fuzzy matching)

`fuzzy_kcat_matching` searches BRENDA (bundled with geckopy) for each EC number,
relaxing the organism, substrate and EC number as needed. `wildcard_level` says
how much the EC number was relaxed; `origin` how close the organism/substrate
match was. *R. toruloides* is not in KEGG, so we pass an empty `PhylDist`
(no phylogenetic-distance weighting) — matching the MATLAB adapter's empty KEGG
ID.

In [ ]:
brenda = load_brenda_data(adapter.get_brenda_db_folder())
kcat_brenda = fuzzy_kcat_matching(ec, brenda, PhylDist())
print(f'BRENDA matches: {len(kcat_brenda)} reactions')
kcat_brenda.head(3)

### Question 4 (cf. MATLAB)

Take the first BRENDA match and look it up on [brenda-enzymes.org](https://www.brenda-enzymes.org).
Why is its `origin` what it is (same/different organism and substrate)?

### Question 5 (cf. MATLAB)

For ~a third of reactions BRENDA finds no kcat (no EC number). What is the
consequence of a missing kcat value for a reaction in the ecModel?

### kcat from DLKcat

DLKcat predicts kcats from substrate SMILES + enzyme sequence. SMILES are
resolved from the cached `smilesDB.tsv` (normally PubChem). The cached
`DLKcat.tsv` already holds the predictions (DLKcat itself runs in Docker).

In [ ]:
find_met_smiles(ec, cache_path='ecRhto/data/smilesDB.tsv')
kcat_dlkcat = read_dlkcat_output(ec, 'ecRhto/data/DLKcat.tsv')
print(f'DLKcat predictions: {len(kcat_dlkcat)} reactions')

### Merge, fill isozymes, standard kcat, and apply

Literature (BRENDA) values are preferred over predictions (DLKcat); the merged
list is written into `ec.ec.kcat`. Isozymes without their own value inherit
one; remaining reactions get a standard kcat. Finally `apply_kcat_constraints`
writes the `MW/kcat` coefficients into the stoichiometric matrix.

In [ ]:
kcat_merged = merge_dlkcat_and_fuzzy_kcats(kcat_dlkcat, kcat_brenda)
apply_kcat_list(ec, kcat_merged)
fill_kcats_from_isozymes(ec)
assign_standard_kcat(ec, uniprot_db)
apply_kcat_constraints(ec)
n_zero = sum(1 for k in ec.ec.kcat if k == 0)
print(f'Reactions still without a kcat: {n_zero}')

eqn = ec.reactions.get_by_id(ec.ec.rxns[0]).reaction
print('Example enzyme-constrained reaction:', ec.ec.rxns[0])
print(' ', eqn)

### Question 6 (cf. MATLAB)

How has the reaction equation changed after `apply_kcat_constraints`? (Look for
the `prot_<id>` pseudo-substrate and its tiny stoichiometric coefficient
`MW/(kcat·3600)`.)

### Protein pool constraint

The total protein available for enzymes is `Ptot · f · sigma`. `Ptot` is the
cellular protein content, `f` the enzyme fraction, `sigma` the average
saturation. This becomes the **upper bound** of `prot_pool_exchange`.

In [ ]:
set_prot_pool_size(ec, p_tot=params.p_tot, f=params.f, sigma=params.sigma)
budget = ec.reactions.get_by_id('prot_pool_exchange').upper_bound
print(f'Protein pool budget (Ptot*f*sigma): {budget:.2f} mg/gDCW')

### Question 7 (cf. MATLAB)

The `f`-factor used here is the adapter default (0.5). What is the consequence
for later simulations of using a different `f` recomputed from proteomics data?

### Question 8 (cf. MATLAB)

What does the protein-pool constraint actually mean, and what is its unit — is
it mmol/gDCW/h like other fluxes, or something else?

## The protein machinery and its direction

The ecModel is built. Look at the protein-pool exchange and an example
enzyme-usage reaction — note the **forward direction and positive bounds**, the
opposite of GECKO MATLAB (see the table at the top).

In [ ]:
ppe = ec.reactions.get_by_id('prot_pool_exchange')
usage = next(r for r in ec.reactions if r.id.startswith('usage_prot_'))
print(f'prot_pool_exchange : {ppe.reaction:28s} bounds={ppe.bounds}')
print(f'{usage.id:19s}: {usage.reaction:28s} bounds={usage.bounds}')

### Question 9 (cf. MATLAB)

Compare the gene–reaction rules of a reaction in the conventional GEM and in the
ecModel (where it is split into `_EXP_1`, `_EXP_2`, … per isoenzyme). What has
happened?

### Question 10 (cf. MATLAB)

If `usage_prot_<id>` carried a flux of 20 mmol/gDCW/h, describe the flux (and
its **sign**) through `prot_pool_exchange`. In geckopy these fluxes are
**positive**; in MATLAB they were negative.

## Stage 3 — model tuning

### Maximum growth rate

With the kcats just assigned, can the model reach the experimental growth rate?

In [ ]:
ec.reactions.get_by_id(params.c_source).lower_bound = -1000   # unconstrained glucose
ec.objective = params.bio_rxn
max_growth = ec.optimize().fluxes[params.bio_rxn]
print(f'Max growth rate of the freshly built ecModel: {max_growth:.4f} /h '
      f'(experimental gR_exp = {params.gr_exp} /h)')

Too low — some kcats are too small, forcing too much protein usage. Inspect the
enzyme usage, then let `sensitivity_tuning` raise the most limiting kcats
(10-fold each) until the model reaches `gr_exp`.

In [ ]:
usage_report = report_enzyme_usage(ec, enzyme_usage(ec, ec.optimize().fluxes))
print(usage_report.top_abs_usage.head(6).to_string())

tuning = sensitivity_tuning(ec)
print(f'Tuned {len(tuning.rxns)} kcat values; growth now '
      f'{ec.optimize().fluxes[params.bio_rxn]:.4f} /h')

### Question 15 (cf. MATLAB)

kcat tuning *raises* selected kcats (less protein needed) while sigma-fitting
*lowers* the protein pool. Describe how these two seem to pull in opposite
directions, and when you would use each.

## Stage 4 — proteomics integration

Load the absolute proteomics (6 conditions × 2 replicates). The file is in
mmol/gDCW; geckopy expects mg/gDCW, so we multiply by the enzyme molecular
weights. We integrate condition **GexpUrea** (exponential growth on glucose) —
the *fifth* condition, index 4 in Python (0-based).

In [ ]:
prot_data = load_prot_data('ecRhto/data/abs_proteomics.txt', repl_per_cond=[2, 2, 2, 2, 2, 2])
mw_by_id = dict(zip(ec.ec.enzymes, ec.ec.mw))
abund = np.asarray(prot_data.abundances, dtype=float).copy()
for i, pid in enumerate(prot_data.uniprot_ids):
    abund[i, :] *= mw_by_id.get(pid, np.nan)      # mmol/gDCW * MW(g/mol) = mg/gDCW
prot_data.abundances = abund

CONDITION = 4
fill_enz_concs(ec, prot_data, data_col=CONDITION)
constrain_enz_concs(ec)
n = int((~np.isnan(np.asarray(ec.ec.concs, dtype=float))).sum())
print(f'Constrained {n} of {len(ec.ec.enzymes)} enzymes with measured concentrations')

In [ ]:
flux_data = load_flux_data('ecRhto/data/fluxData.tsv')
set_prot_pool_size(ec, p_tot=float(np.asarray(flux_data.p_tot)[CONDITION]),
                   f=calculate_f_factor(ec, prot_data))
apply_flux_data_constraints(ec, flux_data, condition=CONDITION,
                            max_min_growth='max', loose_strict_flux='loose')
target = float(np.asarray(flux_data.gr_rate)[CONDITION])
print(f'Growth after proteomics + flux constraints: '
      f'{ec.optimize().objective_value:.4f} /h (experimental {target:.4f})')

The growth rate collapses: some measured enzyme concentrations are too low
(membrane proteins are under-measured). `flexibilize_enz_concs` raises the most
limiting concentrations until the target is reached.

In [ ]:
flex = flexibilize_enz_concs(ec, exp_growth=target, fold_change=10.0)
print(f'Growth after flexibilizing: {ec.optimize().objective_value:.4f} /h '
      f'({len(flex.uniprot_ids)} enzyme concentrations relaxed)')

## Stage 5 — simulation and analysis

### Minimal protein usage

Fix growth at 99% of its maximum and **minimise** the protein pool. Because
`prot_pool_exchange` is a *positive* forward reaction in geckopy, we minimise it
by setting its objective coefficient to `-1` and maximising.

### Question 21 (cf. MATLAB)

In MATLAB you *maximised* `prot_pool_exchange` (a negative-flux reaction) to
minimise protein usage. Why is it the other way around here?

In [ ]:
ec.objective = params.bio_rxn
g = ec.optimize().fluxes[params.bio_rxn]
ec.reactions.get_by_id(params.bio_rxn).lower_bound = 0.99 * g
ec.objective = {ec.reactions.get_by_id('prot_pool_exchange'): -1.0}
print(f'Minimum protein pool usage at 99% of max growth: '
      f'{abs(ec.optimize().fluxes["prot_pool_exchange"]):.2f} mg/gDCW')

### ecModel vs conventional GEM

ecModel reactions are split per isoenzyme/direction, so their fluxes can't be
compared one-to-one with the GEM. `map_rxns_to_conv` maps them back. (Note:
`make_ec_model` modified the GEM in place, so we reload a fresh copy.)

### Question 22 (cf. MATLAB)

Why can the ecModel and conventional-GEM FBA solution vectors not be compared
directly, reaction by reaction?

In [ ]:
gem_fresh = load_conventional_gem(adapter)
gem_fresh.adapter = adapter
ec.objective = params.bio_rxn
mapped = map_rxns_to_conv(ec, gem_fresh, ec.optimize().fluxes)
print('Mapped ecModel fluxes back onto', len(gem_fresh.reactions),
      'conventional reactions:', type(mapped).__name__)

### ec-flux variability analysis (optional)

`ec_fva` collapses the isoenzyme/direction splits and reports one (min, max) per
conventional reaction. It is **slow with the default GLPK solver** (minutes for
thousands of reactions × 2 LPs), so it is left commented out — enable a faster
solver (`ec.solver = "gurobi"`) first.

In [ ]:
# from geckopy import ec_fva
# fva = ec_fva(ec, gem_fresh)        # DataFrame of min_flux / max_flux per reaction
# (fva['max_flux'] - fva['min_flux']).sort_values().plot()  # variability CDF

## Notes on differences from the GECKO MATLAB version

- **Protein-reaction direction (the big one).** geckopy uses the natural
  *forward* direction (positive flux) for `usage_prot_*` and
  `prot_pool_exchange`, the mirror image of MATLAB's reverse/negative
  convention. Minimise (not maximise) the pool for minimal protein usage; apply
  proteomics as an *upper* bound; expect positive usage fluxes.
- **We build the model in Python.** Unlike the first version of this notebook
  (which loaded a MATLAB-written ecModel YAML), here `make_ec_model` builds the
  ecModel from the conventional GEM, so no YAML compatibility workaround is
  needed. To skip the reconstruction you could `save_ec_model`/`load_ec_model` a
  built model instead.
- **BRENDA is bundled** with geckopy (`adapter.get_brenda_db_folder()`); DLKcat
  predictions and SMILES come from the cached `DLKcat.tsv` / `smilesDB.tsv`
  (DLKcat itself runs in Docker). *R. toruloides* is not in KEGG, so an empty
  `PhylDist` is used (no phylogenetic weighting).
- **`make_ec_model` mutates its input GEM** (isoenzyme/reversibility splits), so
  reload a fresh conventional GEM for the ecModel-vs-GEM comparison.
- **Install.** geckopy is alpha and not on PyPI; it is installed from git and
  currently needs `raven-toolbox@main` (its metadata pins `@develop`, which
  lags). The numbers here differ from the MATLAB answer key (different kcat
  merge, solver, and the direction convention) but the qualitative story is the
  same.
- **Solver.** geckopy extends cobrapy and uses GLPK by default; switch with
  `ec.solver = "gurobi"` for speed (FVA in particular).